# Evaluate RARec with Late Fusion

**Mục tiêu**: Đánh giá RARec method sử dụng Late Fusion strategy với threshold-based retrieval

**Phương pháp - Late Fusion:**
1. Với mỗi query, encode bằng SBERT
2. Tính similarity với **TẤT CẢ** câu trong mỗi món ăn
3. Lấy **average similarity** của tất cả câu → điểm số món ăn
4. Filter món ăn có avg_similarity >= threshold
5. Sort theo score giảm dần

**Output format**: JSONL file tương tự các methods khác
```json
{"query_id": 123, "relevant_docs": [{"doc_id": 456, "score": 0.85}, ...]}
```

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv"
RAREC_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec"
EMBEDDINGS_PATH = os.path.join(RAREC_PATH, "recipes_embeddings_list.pkl")
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\eval_ground_truth.jsonl"
OUTPUT_DIR = r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl"

# Similarity threshold for relevant items
SIMILARITY_THRESHOLD = 0.1  # Same threshold as other methods

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"RARec path: {RAREC_PATH}")
print(f"Embeddings path: {EMBEDDINGS_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")

Data path: E:\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv
RARec path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec
Embeddings path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec\recipes_embeddings_list.pkl
Ground truth path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\eval_ground_truth.jsonl
Output directory: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl
Similarity threshold: 0.1


## 1. Load Ground Truth and Data

In [4]:
# Load ground truth
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

query_ids = [item['query_id'] for item in ground_truth]
print(f"Loaded {len(query_ids)} query_ids from ground truth")
print(f"First 5 query_ids: {query_ids[:5]}")

Loaded 200 query_ids from ground truth
First 5 query_ids: [6302, 3779, 768, 3399, 4561]


In [5]:
# Load recipes data
df = pd.read_csv(DATA_PATH)
df['recipe_id'] = df.index  # recipe_id = index
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source', 'recipe_id']


## 2. Load RARec Model Components

In [6]:
# Load Vietnamese SBERT model
print("Loading SBERT model...")
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"Model loaded: {model.get_sentence_embedding_dimension()} dimensions")

Loading SBERT model...
Model loaded: 768 dimensions


In [7]:
# Load recipe embeddings list (Late Fusion - embeddings per sentence)
print("Loading recipe embeddings list...")
with open(EMBEDDINGS_PATH, 'rb') as f:
    recipes_embeddings_list = pickle.load(f)

print(f"Loaded embeddings for {len(recipes_embeddings_list)} recipes")
print(f"  Example: Recipe 0 has {len(recipes_embeddings_list[0])} sentence embeddings")

Loading recipe embeddings list...
Loaded embeddings for 10263 recipes
  Example: Recipe 0 has 5 sentence embeddings


## 3. Implement Late Fusion with Threshold-based Retrieval

In [8]:
def late_fusion_search_with_threshold(query_idx, model, recipes_embeddings_list, df, threshold=0.1, exclude_self=True):
    """
    Late Fusion search với threshold-based retrieval
    
    Strategy:
    1. Encode query recipe (từ df)
    2. Tính similarity với TẤT CẢ câu trong mỗi món
    3. Average similarity → điểm món ăn
    4. Filter món có avg_similarity >= threshold
    5. Exclude self if requested
    6. Sort theo score giảm dần
    
    Args:
        query_idx: Index of query recipe in dataframe
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per recipe
        df: Recipe dataframe
        threshold: Minimum avg similarity to consider relevant
        exclude_self: Whether to exclude query recipe itself
    
    Returns:
        List of {"doc_id": int, "score": float} sorted by score descending
    """
    # 1. Get query recipe text
    query_recipe = df.iloc[query_idx]
    query_text = f"{query_recipe['title']}. {query_recipe['description']}"
    
    # 2. Encode query
    query_embedding = model.encode([query_text])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize
    
    # 3. Calculate average similarity for EACH recipe
    recipe_scores = []
    
    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        # Skip if no embeddings
        if len(dish_embeds) == 0:
            continue
        
        # Skip self if requested
        if exclude_self and recipe_idx == query_idx:
            continue
        
        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)
        
        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()
        
        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)
        
        # Filter by threshold
        if avg_similarity >= threshold:
            recipe_scores.append({
                'doc_id': int(df.iloc[recipe_idx]['recipe_id']),
                'score': float(avg_similarity)
            })
    
    # 4. Sort by score descending
    recipe_scores.sort(key=lambda x: x['score'], reverse=True)
    
    return recipe_scores

In [9]:
# Test the function with a sample query
test_query_idx = 0
print(f"Testing with query index {test_query_idx}:")
print(f"  Query recipe_id: {df.iloc[test_query_idx]['recipe_id']}")
print(f"  Query title: {df.iloc[test_query_idx]['title']}")
print()

test_results = late_fusion_search_with_threshold(
    test_query_idx, 
    model, 
    recipes_embeddings_list, 
    df, 
    threshold=SIMILARITY_THRESHOLD
)

print(f"Found {len(test_results)} relevant recipes (threshold >= {SIMILARITY_THRESHOLD})")
print(f"\nTop 5 results:")
for i, item in enumerate(test_results[:5], 1):
    doc_row = df[df['recipe_id'] == item['doc_id']].iloc[0]
    print(f"  {i}. Doc {item['doc_id']} (score: {item['score']:.4f}) - {doc_row['title']}")

Testing with query index 0:
  Query recipe_id: 0
  Query title: Cách muối dưa hành truyền thống

Found 10200 relevant recipes (threshold >= 0.1)

Top 5 results:
  1. Doc 8707 (score: 0.7362) - Cách muối dưa cải (dưa chua) vàng giòn, để lâu không nổi váng
  2. Doc 9321 (score: 0.7296) - Cách muối dưa hành giòn ngon, không bị hăng đơn giản tại nhà
  3. Doc 619 (score: 0.6962) - Dưa góp kiểu miền Trung
  4. Doc 10057 (score: 0.6953) - Cách muối dưa củ cải giòn ngon không hăng bằng hộp đựng thực phẩm
  5. Doc 5608 (score: 0.6720) - Dưa cải chua (dưa chua) xào thịt ba chỉ siêu ngon cho bữa cơm


## 4. Run Evaluation Loop

In [10]:
def run_rarec_evaluation(query_ids, model, recipes_embeddings_list, df, threshold=0.1):
    """
    Run RARec evaluation with Late Fusion for all queries
    
    Args:
        query_ids: List of query recipe_ids
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per recipe
        df: Recipe dataframe
        threshold: Minimum similarity threshold
    
    Returns:
        List of predictions (one per query)
    """
    predictions = []
    
    # Create recipe_id to index mapping
    recipe_id_to_idx = {recipe_id: idx for idx, recipe_id in enumerate(df['recipe_id'])}
    
    # Track statistics
    total_relevant = 0
    min_relevant = float('inf')
    max_relevant = 0
    
    for query_id in tqdm(query_ids, desc="Evaluating RARec (Late Fusion)"):
        # Get query index
        query_idx = recipe_id_to_idx[query_id]
        
        # Retrieve ALL items >= threshold (excluding self)
        relevant_items = late_fusion_search_with_threshold(
            query_idx,
            model,
            recipes_embeddings_list,
            df,
            threshold=threshold,
            exclude_self=True
        )
        
        # Update statistics
        num_relevant = len(relevant_items)
        total_relevant += num_relevant
        min_relevant = min(min_relevant, num_relevant)
        max_relevant = max(max_relevant, num_relevant)
        
        # Create prediction record
        pred_record = {
            "query_id": int(query_id),
            "relevant_docs": relevant_items  # All items >= threshold
        }
        predictions.append(pred_record)
    
    # Print statistics
    avg_relevant = total_relevant / len(query_ids)
    print(f"\nStatistics for RARec (Late Fusion):")
    print(f"  Average relevant items per query: {avg_relevant:.2f}")
    print(f"  Min relevant items: {min_relevant}")
    print(f"  Max relevant items: {max_relevant}")
    print(f"  Total relevant pairs: {total_relevant}")
    
    return predictions

In [11]:
# Test with small subset first (5 queries)
test_predictions = run_rarec_evaluation(
    query_ids[:5], 
    model, 
    recipes_embeddings_list, 
    df, 
    threshold=SIMILARITY_THRESHOLD
)

print(f"\nExample prediction:")
print(f"  Query ID: {test_predictions[0]['query_id']}")
print(f"  Number of relevant docs: {len(test_predictions[0]['relevant_docs'])}")
print(f"  Top 5 relevant docs:")
for doc in test_predictions[0]['relevant_docs'][:5]:
    print(f"    - Doc {doc['doc_id']}: {doc['score']:.4f}")

Evaluating RARec (Late Fusion): 100%|██████████| 5/5 [00:07<00:00,  1.41s/it]


Statistics for RARec (Late Fusion):
  Average relevant items per query: 10261.80
  Min relevant items: 10261
  Max relevant items: 10262
  Total relevant pairs: 51309

Example prediction:
  Query ID: 6302
  Number of relevant docs: 10262
  Top 5 relevant docs:
    - Doc 6395: 0.7110
    - Doc 6317: 0.6997
    - Doc 6123: 0.6980
    - Doc 298: 0.6884
    - Doc 6381: 0.6847


In [12]:
# Run evaluation for ALL 200 queries
print(f"Threshold: {SIMILARITY_THRESHOLD}")

all_predictions = run_rarec_evaluation(
    query_ids,
    model,
    recipes_embeddings_list,
    df,
    threshold=SIMILARITY_THRESHOLD
)

Threshold: 0.1


Evaluating RARec (Late Fusion): 100%|██████████| 200/200 [04:11<00:00,  1.26s/it]


Statistics for RARec (Late Fusion):
  Average relevant items per query: 10261.48
  Min relevant items: 10242
  Max relevant items: 10262
  Total relevant pairs: 2052296


## 5. Save Predictions to JSONL

In [13]:
# Save predictions to JSONL file
output_file = os.path.join(OUTPUT_DIR, "RARec_Late_Fusion_pred.jsonl")

with open(output_file, 'w', encoding='utf-8') as f:
    for pred in all_predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + '\n')

print(f"Saved {len(all_predictions)} predictions to {output_file}")
print(f"  Total relevant pairs: {sum(len(p['relevant_docs']) for p in all_predictions)}")

# Calculate average
total_pairs = sum(len(p['relevant_docs']) for p in all_predictions)
avg_pairs = total_pairs / len(query_ids)
print(f"  Average pairs per query: {avg_pairs:.2f}")

Saved 200 predictions to E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl\RARec_Late_Fusion_pred.jsonl
  Total relevant pairs: 2052296
  Average pairs per query: 10261.48


## 6. Sanity Checks

### 6.1. Verify no self-recommendations

In [14]:
# Verify no self-recommendations
violations = []
for pred in all_predictions:
    query_id = pred['query_id']
    doc_ids = [item['doc_id'] for item in pred['relevant_docs']]
    
    if query_id in doc_ids:
        violations.append(query_id)

if violations:
    print(f"Found {len(violations)} self-recommendations!")
    print(f"   Query IDs with self-recommendation: {violations[:5]}...")
else:
    print(f"No self-recommendations found ({len(all_predictions)} queries checked)")

No self-recommendations found (200 queries checked)


### 6.2. Score distribution

In [15]:
# Analyze relevant items per query
sizes = [len(p['relevant_docs']) for p in all_predictions]

print("RARec Late Fusion - Relevant items per query:")
print(f"  Min: {min(sizes)}")
print(f"  Max: {max(sizes)}")
print(f"  Mean: {np.mean(sizes):.2f}")
print(f"  Median: {np.median(sizes):.2f}")
print(f"  Std: {np.std(sizes):.2f}")

print(f"\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    val = np.percentile(sizes, p)
    print(f"  {p}th: {val:.0f}")

RARec Late Fusion - Relevant items per query:
  Min: 10242
  Max: 10262
  Mean: 10261.48
  Median: 10262.00
  Std: 2.31

Percentiles:
  25th: 10262
  50th: 10262
  75th: 10262
  90th: 10262
  95th: 10262
  99th: 10262


In [16]:
# Analyze score distribution
all_scores = []
for pred in all_predictions:
    all_scores.extend([item['score'] for item in pred['relevant_docs']])

if len(all_scores) > 0:
    print(f"\nRARec Late Fusion - Score distribution ({len(all_scores)} total pairs):")
    print(f"  Score range: [{min(all_scores):.4f}, {max(all_scores):.4f}]")
    print(f"  Mean: {np.mean(all_scores):.4f}")
    print(f"  Median: {np.median(all_scores):.4f}")
    
    print(f"\nScores by threshold:")
    for thresh in [0.5, 0.3, 0.2, 0.1, 0.05]:
        count = sum(1 for s in all_scores if s >= thresh)
        pct = 100 * count / len(all_scores)
        print(f"  >= {thresh:.2f}: {count:6d} ({pct:5.2f}%)")


RARec Late Fusion - Score distribution (2052296 total pairs):
  Score range: [0.1004, 0.7896]
  Mean: 0.4037
  Median: 0.4033

Scores by threshold:
  >= 0.50: 277487 (13.52%)
  >= 0.30: 1811876 (88.29%)
  >= 0.20: 2039574 (99.38%)
  >= 0.10: 2052296 (100.00%)
  >= 0.05: 2052296 (100.00%)
